# 04 部署发布

**目标**：把训练好的模型变成一个能被外部调用的服务。

两种发布形态：
1. **REST API**（FastAPI）—— 供其他系统调用
2. **交互页面**（Streamlit）—— 供业务人员直接使用

In [ ]:
import json
import joblib
import pandas as pd

prep = joblib.load('../../models/preprocessor.pkl')
model = joblib.load('../../models/model.pkl')
print('预处理器:', type(prep).__name__)
print('模型    :', type(model).__name__)
print(json.dumps(json.load(open('../../models/metrics.json', encoding='utf-8')),
                 ensure_ascii=False, indent=2))

## 1. 本地推理：先确认模型能对单条样本给出合理结果

In [ ]:
row = {
    'age': 29, 'tenure_months': 3, 'monthly_charge': 130.0, 'total_charge': 390.0,
    'contract_type': '按月付费', 'payment_method': '电子支票',
    'support_calls': 9, 'is_active': 0,
}
row['avg_monthly'] = row['total_charge'] / row['tenure_months']
row['support_per_year'] = row['support_calls'] / (row['tenure_months'] / 12)

NUM_COLS = ['age', 'tenure_months', 'monthly_charge', 'total_charge',
            'support_calls', 'is_active', 'avg_monthly', 'support_per_year']
CAT_COLS = ['contract_type', 'payment_method']

X = prep.transform(pd.DataFrame([row])[NUM_COLS + CAT_COLS])
print(f'流失概率: {model.predict_proba(X)[0, 1]:.2%}')

## 2. 启动 REST 服务

**先确认是否已经在跑**：浏览器打开 <http://localhost:8000/docs> ，能打开就不用启动
（`start.bat` 启动时会自动拉起这个服务）。

如果确实没起，在**编辑器底部的终端**里执行——提示符是 `coder@...:/project$`，
这是容器内的终端：

```bash
cd /home/coder/project
uvicorn src.serve:app --host 0.0.0.0 --port 8000
```

> **注意**：`uvicorn` 装在容器的 `/opt/venv/bin` 里。
> 如果你在 Windows 的命令行窗口里执行（提示符是 `C:\...>`），会报
> `uvicorn : command not found` —— 那是宿主机，没有这个命令。
>
> 从宿主机启动要这样写：
>
> ```bat
> docker exec -d ai-lab bash -c "cd /home/coder/project && /opt/venv/bin/python -m uvicorn src.serve:app --host 0.0.0.0 --port 8000"
> ```

然后再打开 <http://localhost:8000/docs> ，可以直接在页面上调试接口。


In [1]:
import requests

payload = {
    'age': 29, 'tenure_months': 3, 'monthly_charge': 130.0, 'total_charge': 390.0,
    'contract_type': '按月付费', 'payment_method': '电子支票',
    'support_calls': 9, 'is_active': 0,
}
try:
    r = requests.post('http://localhost:8000/predict', json=payload, timeout=5)
    print(r.status_code, r.json())
except requests.exceptions.ConnectionError:
    print('服务未启动，请先在终端执行 uvicorn src.serve:app --port 8000')

200 {'churn': 1, 'probability': 0.9199, 'risk_level': '高'}


## 3. 启动交互页面

```bash
streamlit run apps/streamlit_app.py --server.address 0.0.0.0 --server.port 8501
```
打开 http://localhost:8501 ，拖动滑块即可看到概率实时变化。

## 小结

- 部署的本质是「让别人不用懂模型也能用模型」
- REST API 适合系统集成，交互页面适合业务演示，两者共用同一份 `model.pkl`
- 到这一步，一条从原始数据到线上服务的链路就闭环了